[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [1]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [2]:
data_frame = pd.read_csv("Dropped_match_data.csv")

#### Deleting unwanted features
Some features are not needed or are less workable than others, removing them cleans up the data and provides easier, simpler ways to manipualate the data present

In [ ]:
Dropped_match_data_v5 = data_frame.drop(
    columns=[
        "blueTeamControlWardsPlaced",
        "blueTeamWardsPlaced",
        "blueTeamDragonKills",
        "blueTeamHeraldKills",
        "blueTeamInhibitorsDestroyed",
        "blueTeamTurretPlatesDestroyed",
        "blueTeamJungleMinions",
        "blueTeamTotalDamageToChamps",
        "redTeamControlWardsPlaced",
        "redTeamWardsPlaced",
        "redTeamDragonKills",
        "redTeamHeraldKills",
        "redTeamInhibitorsDestroyed",
        "redTeamTurretPlatesDestroyed",
        "redTeamJungleMinions",
        "redTeamTotalDamageToChamps",
    ],
    errors="ignore",
)

Dropped_match_data_v5.to_csv(
    "../2.1.Data_Wrangling/Dropped_data.csv", index=False, header=True
)

#### Remove Duplicates

Duplicate data can have detrimental effects on your machine learning models and outcomes, such as reducing data diversity and representativeness, which can lead to overfitting or biased models.

The `duplicated().sum()` method call returns the count of duplicate rows in the data frame.

In [ ]:
data_frame.duplicated().sum()

The `drop_duplicates()` method call can be then stored back onto the data_frame variable removing the duplicates.

In [ ]:
data_frame = data_frame.drop_duplicates()
data_frame.duplicated().sum()

We can check that there are no data entry errors by the `unique()` method call.

In [ ]:
data_frame['blueTeamTotalKills'].unique()

#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [ ]:
#get the inter-quartile range on the feature column
print(data_frame['blueTeamTotalKills'].describe())
Q1 = data_frame['blueTeamTotalKills'].quantile(0.25)
Q3 = data_frame['blueTeamTotalKills'].quantile(0.75)
IQR = Q3 - Q1
print(f'Outliers are a blueTeamTotalKills above {Q3 + IQR * 1.5} or below {Q1 - IQR * 1.5}')


count    24035.000000
mean        12.669856
std          4.723191
min          0.000000
25%          9.000000
50%         12.000000
75%         16.000000
max         26.000000
Name: blueTeamTotalKills, dtype: float64
Outliers are a blueTeamTotalKills above 26.5 or below -1.5


In [4]:
# Filter  blueTeamTotalKills within the acceptable range
data_frame = data_frame[
    (data_frame["blueTeamTotalKills"] >= Q1 - 1.5 * IQR)
    & (data_frame["blueTeamTotalKills"] <= Q3 + 1.5 * IQR)
]
print(data_frame["blueTeamTotalKills"].describe())

count    24035.000000
mean        12.669856
std          4.723191
min          0.000000
25%          9.000000
50%         12.000000
75%         16.000000
max         26.000000
Name: blueTeamTotalKills, dtype: float64


#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [ ]:
scale_feature = 'blueTeamTotalKills'

# the minimum value with space for outliers
MIN_blueTeamTotalKills = 0

# the maximum value with space for outliers
MAX_blueTeamTotalKills = 26

# scale features
data_frame[scale_feature] = [
    (X - MIN_blueTeamTotalKills) / (MAX_blueTeamTotalKills - MIN_blueTeamTotalKills)
    for X in data_frame[scale_feature]
]

data_frame.describe()

#### Scaling Multiple features can take a while.
MinMaxScaler can be used to scale multiple features into [0,1] which is crucial for Logisitic regression, SVM, KNN, or neural networks. MinMaxScaler automatically computes min/max values from [The match data](/workspaces/2026SE_MLOPSTask_Kelvin.A/2.Model_Development/2.1.Data_Wrangling/Dropped_match_data.csv), and handles all features. the target blueWin is ignored as that is the final outcome, and is already binary.

In [7]:
from sklearn.preprocessing import MinMaxScaler

features = [
    "blueTeamTotalKills",
    "blueTeamTowersDestroyed",
    "blueTeamFirstBlood",
    "blueTeamMinionsKilled",
    "blueTeamTotalGold",
    "blueTeamXp",
    "redTeamTotalKills",
    "redTeamTowersDestroyed",
    "redTeamMinionsKilled",
    "redTeamTotalGold",
    "redTeamXp",
]

scaler = MinMaxScaler()
data_frame[features] = scaler.fit_transform(data_frame[features])

data_frame.to_csv(
    "../2.2.Feature_Engineering/2.2.1.wrangled_data.csv", index=False, header=True
)

print(f"Pre-scaled minimum: {data_frame[features].min()}")
print(f"Pre-scaled minimum: {data_frame[features].max()}")
print(f"Scaled minimum: {scaler.data_min_}")
print(f"Scaled maximum: {scaler.data_max_}")

Pre-scaled minimum: blueTeamTotalKills         0.0
blueTeamTowersDestroyed    0.0
blueTeamFirstBlood         0.0
blueTeamMinionsKilled      0.0
blueTeamTotalGold          0.0
blueTeamXp                 0.0
redTeamTotalKills          0.0
redTeamTowersDestroyed     0.0
redTeamMinionsKilled       0.0
redTeamTotalGold           0.0
redTeamXp                  0.0
dtype: float64
Pre-scaled minimum: blueTeamTotalKills         1.0
blueTeamTowersDestroyed    1.0
blueTeamFirstBlood         1.0
blueTeamMinionsKilled      1.0
blueTeamTotalGold          1.0
blueTeamXp                 1.0
redTeamTotalKills          1.0
redTeamTowersDestroyed     1.0
redTeamMinionsKilled       1.0
redTeamTotalGold           1.0
redTeamXp                  1.0
dtype: float64
Scaled minimum: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Scaled maximum: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [ ]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False, headers=True)